# 2.5 Markov Chains

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 2 — Matrices**

---

### What you will learn

- What a **stochastic matrix** is and why its columns must sum to 1.
- How to represent a **state vector** and evolve it through time via matrix multiplication.
- How to compute **matrix powers** $P^k$ to predict long-term behavior.
- How Markov chains **converge** to a unique **steady-state vector**.
- How to find the steady state **algebraically** by solving $(P - I)\mathbf{x} = \mathbf{0}$.
- What a **regular** Markov chain is and why regularity guarantees convergence.

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.ops import add, scalar_mul, multiply, transpose
from linalg_core.elimination import solve
from linalg_core import EPSILON

import numpy as np
import matplotlib.pyplot as plt
import math

---

## 2. Motivation — Weather Prediction

A meteorologist models the weather in a town with two states: **Sunny** and **Rainy**.
By analyzing historical data, she finds the following transition probabilities:

| | Tomorrow Sunny | Tomorrow Rainy |
|---|---|---|
| **Today Sunny** | 0.70 | 0.30 |
| **Today Rainy** | 0.40 | 0.60 |

In other words:
- If today is **sunny**, there is a 70% chance tomorrow is sunny and a 30% chance of rain.
- If today is **rainy**, there is a 40% chance tomorrow is sunny and a 60% chance of rain.

We encode this as a **transition matrix** $P$ where column $j$ represents the transition
probabilities *from* state $j$:

$$P = \begin{bmatrix} 0.7 & 0.4 \\ 0.3 & 0.6 \end{bmatrix}$$

**Questions:**
1. If today is sunny, what is the probability distribution after **7 days**?
2. What about after **100 days**? Does it matter whether we started sunny or rainy?

These are the central questions of **Markov chain** theory.

---

## 3. Build — Core Concepts

### 3.1 Stochastic Matrices

> **Definition (Larson, Sec. 2.5).** A square matrix $P$ is called a **stochastic matrix**
> (or **transition matrix**) if:
>
> 1. Every entry is **non-negative**: $p_{ij} \ge 0$ for all $i, j$.
> 2. Every **column sums to 1**: $\displaystyle\sum_{i=1}^{n} p_{ij} = 1$ for each column $j$.

Each column of $P$ is a probability distribution — it describes the probabilities of
transitioning *from* a given state *to* all possible states. The column-sum constraint
ensures that the total probability leaving any state is exactly 1.

Let us implement a function to verify this property.

In [ ]:
def is_stochastic(P):
    """Check whether P is a (column) stochastic matrix.

    Returns True if every entry is non-negative and every column sums to 1.
    """
    if P.rows != P.cols:
        return False
    n = P.rows
    for i in range(n):
        for j in range(n):
            if P[i, j] < -EPSILON:
                return False
    for j in range(n):
        col_sum = sum(P[i, j] for i in range(n))
        if abs(col_sum - 1.0) > EPSILON:
            return False
    return True


P = Matrix.from_lists([
    [0.7, 0.4],
    [0.3, 0.6],
])

print("Transition matrix P:")
print(P)
print(f"\nColumn 0 sum: {P[0,0] + P[1,0]:.1f}")
print(f"Column 1 sum: {P[0,1] + P[1,1]:.1f}")
print(f"\nis_stochastic(P): {is_stochastic(P)}")

bad = Matrix.from_lists([[0.7, 0.5], [0.3, 0.6]])
print(f"\nNon-stochastic example (col 1 sums to 1.1): is_stochastic = {is_stochastic(bad)}")

### 3.2 State Vectors

> **Definition.** A **state vector** $\mathbf{x}$ is a column vector whose entries are
> non-negative and sum to 1. It represents a probability distribution over the states.

For our weather model, the state vector has two entries:
$\mathbf{x} = \begin{bmatrix} p(\text{sunny}) \\ p(\text{rainy}) \end{bmatrix}$.

Since we know today is sunny, our initial state vector is:

$$\mathbf{x}_0 = \begin{bmatrix} 1 \\ 0 \end{bmatrix}$$

In [ ]:
x0 = Matrix.from_vector([1.0, 0.0])

print("Initial state vector x_0 (sunny today):")
print(x0)
print(f"\nEntries sum to: {x0[0,0] + x0[1,0]:.1f}")

### 3.3 One Step — The Markov Update

The **fundamental equation** of a Markov chain is:

$$\mathbf{x}_{k+1} = P \, \mathbf{x}_k$$

Multiplying the transition matrix $P$ by the current state vector $\mathbf{x}_k$ produces
the probability distribution one time step later.

Let us compute $\mathbf{x}_1 = P \mathbf{x}_0$ to find tomorrow's weather probabilities.

In [ ]:
x1 = multiply(P, x0)

print("x_1 = P * x_0:")
print(x1)
print(f"\nTomorrow: {x1[0,0]*100:.0f}% sunny, {x1[1,0]*100:.0f}% rainy")
print(f"Entries sum to: {x1[0,0] + x1[1,0]:.4f}")

print("\nStep-by-step:")
print(f"  p(sunny) = 0.7 * {x0[0,0]:.0f} + 0.4 * {x0[1,0]:.0f} = {x1[0,0]:.2f}")
print(f"  p(rainy) = 0.3 * {x0[0,0]:.0f} + 0.6 * {x0[1,0]:.0f} = {x1[1,0]:.2f}")

### 3.4 Iterating — Matrix Powers

After $k$ steps, the state vector is:

$$\mathbf{x}_k = P^k \, \mathbf{x}_0$$

This follows from applying $\mathbf{x}_{k+1} = P \mathbf{x}_k$ repeatedly:

$$\mathbf{x}_1 = P \mathbf{x}_0, \quad \mathbf{x}_2 = P \mathbf{x}_1 = P^2 \mathbf{x}_0, \quad \ldots, \quad \mathbf{x}_k = P^k \mathbf{x}_0$$

We implement `matrix_power(A, n)` via repeated multiplication and use it to find
the state after 7 days.

In [ ]:
def matrix_power(A, n):
    """Compute A^n by repeated multiplication.

    A^0 = I (identity), A^n = A * A^(n-1).
    """
    if A.rows != A.cols:
        raise ValueError("Matrix power requires a square matrix")
    if n < 0:
        raise ValueError("Negative powers not supported")
    result = Matrix.identity(A.rows)
    for _ in range(n):
        result = multiply(result, A)
    return result


print("State vectors x_0 through x_7:")
print(f"{'Step':>4}  {'p(sunny)':>10}  {'p(rainy)':>10}  {'sum':>6}")
print("-" * 38)

xk = x0
for k in range(8):
    s = xk[0, 0] + xk[1, 0]
    print(f"{k:>4}  {xk[0,0]:>10.6f}  {xk[1,0]:>10.6f}  {s:>6.4f}")
    xk = multiply(P, xk)

print(f"\nAfter 7 days: {x0[0,0]*100:.0f}% sunny start -> "
      f"{matrix_power(P, 7)[0,0] * x0[0,0] + matrix_power(P, 7)[0,1] * x0[1,0]:.4f} sunny")

### 3.5 Convergence — Finding the Steady State Iteratively

A remarkable property of many Markov chains is that the state vector **converges** to
a fixed vector $\mathbf{q}$ as $k \to \infty$, regardless of the initial state:

$$\lim_{k \to \infty} P^k \mathbf{x}_0 = \mathbf{q}$$

This $\mathbf{q}$ is called the **steady-state vector** (or **stationary distribution**).
It satisfies $P\mathbf{q} = \mathbf{q}$ — applying the transition one more time
does not change the distribution.

We iterate until $\|\mathbf{x}_{k+1} - \mathbf{x}_k\| < \varepsilon$ to find it numerically.

In [ ]:
def vec_norm(v):
    """Compute the Euclidean norm of a column vector (Matrix with 1 column)."""
    return math.sqrt(sum(v[i, 0] ** 2 for i in range(v.rows)))


def find_steady_state_iterative(P, x0, tol=1e-10, max_iter=1000):
    """Iterate x_{k+1} = P * x_k until convergence.

    Returns (steady_state, num_iterations, history).
    """
    xk = x0
    history = [(xk[0, 0], xk[1, 0])]
    for k in range(max_iter):
        xk1 = multiply(P, xk)
        diff = add(xk1, scalar_mul(xk, -1))
        history.append((xk1[0, 0], xk1[1, 0]))
        if vec_norm(diff) < tol:
            return xk1, k + 1, history
        xk = xk1
    return xk, max_iter, history


q_iter, num_iter, history = find_steady_state_iterative(P, x0)

print("Iterative steady-state search:")
print(f"  Converged in {num_iter} iterations")
print(f"\nSteady-state vector q:")
print(q_iter)
print(f"\n  p(sunny) = {q_iter[0,0]:.6f}")
print(f"  p(rainy) = {q_iter[1,0]:.6f}")
print(f"  sum      = {q_iter[0,0] + q_iter[1,0]:.6f}")

Pq = multiply(P, q_iter)
check_diff = add(Pq, scalar_mul(q_iter, -1))
print(f"\nVerification: ||Pq - q|| = {vec_norm(check_diff):.2e}  (should be ~0)")

### 3.6 Algebraic Steady State — Solving $(P - I)\mathbf{x} = \mathbf{0}$

The steady-state vector $\mathbf{q}$ satisfies $P\mathbf{q} = \mathbf{q}$, which we can rewrite as:

$$(P - I)\mathbf{q} = \mathbf{0}$$

This is a homogeneous system. Since $P$ is stochastic, the columns of $P - I$ sum to 0,
so the system always has a nontrivial solution. But the solution has a **free variable**
(the overall scale), so we add the constraint that entries sum to 1:

$$q_1 + q_2 + \cdots + q_n = 1$$

We augment the system $\begin{bmatrix} P - I \\ \mathbf{1}^T \end{bmatrix} \mathbf{q} = \begin{bmatrix} \mathbf{0} \\ 1 \end{bmatrix}$ and solve.

In [ ]:
n = P.rows
I = Matrix.identity(n)
P_minus_I = add(P, scalar_mul(I, -1))

print("P - I =")
print(P_minus_I)

aug_rows = n + 1
aug_cols = n + 1
aug = Matrix(aug_rows, aug_cols)

for i in range(n):
    for j in range(n):
        aug[i, j] = P_minus_I[i, j]
    aug[i, n] = 0.0

for j in range(n):
    aug[n, j] = 1.0
aug[n, n] = 1.0

print("\nAugmented system [(P-I) | 0; 1^T | 1]:")
print(aug)

sol_type, sol_result = solve(aug)
print(f"\nSolution type: {sol_type}")
print(f"Solution: {sol_result}")

q_algebraic = Matrix.from_vector(sol_result)
print(f"\nAlgebraic steady-state vector:")
print(q_algebraic)
print(f"\n  p(sunny) = {q_algebraic[0,0]:.6f}")
print(f"  p(rainy) = {q_algebraic[1,0]:.6f}")
print(f"  sum      = {q_algebraic[0,0] + q_algebraic[1,0]:.6f}")

In [ ]:
diff = add(q_iter, scalar_mul(q_algebraic, -1))
print("Comparison: iterative vs algebraic steady state")
print(f"  Iterative:  sunny={q_iter[0,0]:.10f}  rainy={q_iter[1,0]:.10f}")
print(f"  Algebraic:  sunny={q_algebraic[0,0]:.10f}  rainy={q_algebraic[1,0]:.10f}")
print(f"  ||diff||  = {vec_norm(diff):.2e}")
print(f"\nThe two methods agree to high precision.")

print(f"\nInterpretation: In the long run, it is sunny about {q_algebraic[0,0]*100:.1f}% of the time")
print(f"and rainy about {q_algebraic[1,0]*100:.1f}% of the time, regardless of today's weather.")

### 3.7 Regular Markov Chains — Guaranteed Convergence

> **Definition (Larson, Sec. 2.5).** A stochastic matrix $P$ is called **regular** if
> some power $P^k$ has **all positive entries** (no zeros).

> **Theorem.** If $P$ is a regular stochastic matrix, then:
> 1. $P$ has a **unique** steady-state vector $\mathbf{q}$.
> 2. The sequence $P^k \mathbf{x}_0$ converges to $\mathbf{q}$ for **any** initial state vector $\mathbf{x}_0$.
> 3. The columns of $P^k$ all converge to $\mathbf{q}$ as $k \to \infty$.

This is a powerful result: regularity guarantees that the long-run behavior
is completely determined by $P$ alone — the initial conditions are forgotten.

Let us verify that our weather matrix is regular, and demonstrate convergence
from two different starting states.

In [ ]:
def is_all_positive(M):
    """Check if every entry of M is strictly positive."""
    return all(M.data[k] > EPSILON for k in range(len(M.data)))


def is_regular(P, max_power=50):
    """Check if P is a regular stochastic matrix.

    Tests whether some P^k (k = 1..max_power) has all positive entries.
    Returns (is_regular, power_needed).
    """
    Pk = P
    for k in range(1, max_power + 1):
        if is_all_positive(Pk):
            return True, k
        Pk = multiply(Pk, P)
    return False, -1


regular, power = is_regular(P)
print(f"Is P regular? {regular}  (all positive at P^{power})")
print(f"\nP^{power} =")
print(matrix_power(P, power))

print("\n" + "=" * 55)
print("Convergence from two different initial states:")
print("=" * 55)

x0_sunny = Matrix.from_vector([1.0, 0.0])
x0_rainy = Matrix.from_vector([0.0, 1.0])

q_sunny, n_sunny, hist_sunny = find_steady_state_iterative(P, x0_sunny)
q_rainy, n_rainy, hist_rainy = find_steady_state_iterative(P, x0_rainy)

print(f"\nStarting sunny [1, 0]:")
print(f"  Converged in {n_sunny} iterations")
print(f"  q = [{q_sunny[0,0]:.6f}, {q_sunny[1,0]:.6f}]")

print(f"\nStarting rainy [0, 1]:")
print(f"  Converged in {n_rainy} iterations")
print(f"  q = [{q_rainy[0,0]:.6f}, {q_rainy[1,0]:.6f}]")

diff = add(q_sunny, scalar_mul(q_rainy, -1))
print(f"\n||q_sunny - q_rainy|| = {vec_norm(diff):.2e}")
print("Both converge to the same steady state, confirming the theorem.")

---

## 4. Verify — Cross-Check with NumPy

We verify our from-scratch implementations against NumPy to ensure correctness.

In [ ]:
def to_np(mat):
    """Convert our Matrix to a NumPy array."""
    return np.array(mat.to_lists())


def check_match(ours, theirs, label):
    """Compare our Matrix result against a NumPy array."""
    ours_np = to_np(ours)
    match = np.allclose(ours_np, theirs, atol=1e-9)
    status = "PASS" if match else "FAIL"
    print(f"[{status}] {label}")
    if not match:
        print(f"  Ours:   {ours_np.flatten()}")
        print(f"  NumPy:  {theirs.flatten()}")
    return match

In [ ]:
P_np = to_np(P)
x0_np = to_np(x0)

print("=" * 55)
print("VERIFICATION: Stochastic matrix properties")
print("=" * 55)

col_sums = P_np.sum(axis=0)
print(f"[{'PASS' if np.allclose(col_sums, 1.0) else 'FAIL'}] Column sums = {col_sums} (should be [1, 1])")
print(f"[{'PASS' if (P_np >= 0).all() else 'FAIL'}] All entries non-negative")

print()
print("=" * 55)
print("VERIFICATION: State vector sums to 1 at each step")
print("=" * 55)

xk_np = x0_np.copy()
all_sum_ok = True
for k in range(20):
    s = xk_np.sum()
    if not np.isclose(s, 1.0):
        print(f"  Step {k}: sum = {s:.10f} -- FAIL")
        all_sum_ok = False
    xk_np = P_np @ xk_np
print(f"[{'PASS' if all_sum_ok else 'FAIL'}] State vector sums to 1 at every step (0..19)")

In [ ]:
print("=" * 55)
print("VERIFICATION: matrix_power vs np.linalg.matrix_power")
print("=" * 55)

for k in [1, 5, 7, 10, 50, 100]:
    ours = matrix_power(P, k)
    theirs = np.linalg.matrix_power(P_np, k)
    check_match(ours, theirs, f"P^{k}")

In [ ]:
print("=" * 55)
print("VERIFICATION: Iterative vs algebraic steady state")
print("=" * 55)

q_iter_np = to_np(q_iter)
q_alg_np = to_np(q_algebraic)

match = np.allclose(q_iter_np, q_alg_np, atol=1e-8)
print(f"[{'PASS' if match else 'FAIL'}] Iterative steady state matches algebraic")
print(f"  Iterative:  {q_iter_np.flatten()}")
print(f"  Algebraic:  {q_alg_np.flatten()}")

print()
print("=" * 55)
print("VERIFICATION: P^100 columns converge to steady state")
print("=" * 55)

P100 = matrix_power(P, 100)
P100_np = np.linalg.matrix_power(P_np, 100)

check_match(P100, P100_np, "P^100 matches NumPy")

col0_match = np.allclose(to_np(P100)[:, 0], q_alg_np.flatten(), atol=1e-8)
col1_match = np.allclose(to_np(P100)[:, 1], q_alg_np.flatten(), atol=1e-8)
print(f"[{'PASS' if col0_match else 'FAIL'}] P^100 column 0 matches steady state")
print(f"[{'PASS' if col1_match else 'FAIL'}] P^100 column 1 matches steady state")

---

## 5. Visualize — Convergence to Steady State

We plot the probability of each state over time, starting from $\mathbf{x}_0 = [1, 0]^T$
(sunny today). The horizontal dashed lines show the algebraic steady-state values.

In [ ]:
num_steps = 50
xk = x0
sunny_probs = [xk[0, 0]]
rainy_probs = [xk[1, 0]]

for k in range(num_steps):
    xk = multiply(P, xk)
    sunny_probs.append(xk[0, 0])
    rainy_probs.append(xk[1, 0])

steps = list(range(num_steps + 1))

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(steps, sunny_probs, 'o-', color='goldenrod', markersize=3,
        linewidth=1.5, label='p(Sunny)')
ax.plot(steps, rainy_probs, 's-', color='steelblue', markersize=3,
        linewidth=1.5, label='p(Rainy)')

q_sunny_val = q_algebraic[0, 0]
q_rainy_val = q_algebraic[1, 0]

ax.axhline(y=q_sunny_val, color='goldenrod', linestyle='--', alpha=0.7,
           label=f'Steady state (sunny) = {q_sunny_val:.4f}')
ax.axhline(y=q_rainy_val, color='steelblue', linestyle='--', alpha=0.7,
           label=f'Steady state (rainy) = {q_rainy_val:.4f}')

ax.set_xlabel('Time Step $k$', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Markov Chain Convergence: Weather Model\n'
             r'$\mathbf{x}_0 = [1,\, 0]^T$ (sunny today)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='center right')
ax.set_xlim(-0.5, num_steps + 0.5)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 6. Exercises

Test your understanding with the following problems. Write your solutions
in the code cells provided.

### Exercise 1 (Easy) — Brand Switching

A market has two competing brands, **A** and **B**. Each month:
- 80% of brand A customers stay with A, 20% switch to B.
- 30% of brand B customers switch to A, 70% stay with B.

1. Write the $2 \times 2$ transition matrix $P$ and verify it is stochastic.
2. If brand A currently has 60% market share ($\mathbf{x}_0 = [0.6, 0.4]^T$), find the
   market share after 1, 5, and 10 months.
3. Find the steady-state market share using either the iterative or algebraic method.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium) — Three-State Weather

Extend the weather model to three states: **Sunny**, **Cloudy**, and **Rainy**.
The transition matrix is:

$$P = \begin{bmatrix}
0.5 & 0.3 & 0.2 \\
0.3 & 0.4 & 0.3 \\
0.2 & 0.3 & 0.5
\end{bmatrix}$$

1. Verify that $P$ is stochastic using `is_stochastic`.
2. Starting from $\mathbf{x}_0 = [1, 0, 0]^T$ (sunny today), iterate for 20 steps
   and print the state vector at each step.
3. Find the algebraic steady state by solving the augmented system
   $\begin{bmatrix} P - I \\ \mathbf{1}^T \end{bmatrix} \mathbf{q} = \begin{bmatrix} \mathbf{0} \\ 1 \end{bmatrix}$.
4. Check if $P$ is regular using `is_regular`.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge) — Absorbing Markov Chain

A state $i$ is called **absorbing** if $p_{ii} = 1$ (once entered, you never leave).
A Markov chain is **absorbing** if it has at least one absorbing state and every
state can eventually reach an absorbing state.

Consider a simple board game with 4 positions. Position 4 is the **finish** (absorbing).
From positions 1, 2, 3, you move forward one or two spaces with equal probability:

$$P = \begin{bmatrix}
0   & 0   & 0   & 0 \\
0.5 & 0   & 0   & 0 \\
0.5 & 0.5 & 0   & 0 \\
0   & 0.5 & 1.0 & 1
\end{bmatrix}$$

1. Verify $P$ is stochastic.
2. Starting at position 1 ($\mathbf{x}_0 = [1, 0, 0, 0]^T$), iterate until
   $p(\text{finish}) > 0.99$. How many steps does it take?
3. Is this chain regular? Why or why not?
4. Compute $P^{20}$ and examine its columns. What do you observe about
   absorbing chains vs regular chains?

*Hint:* An absorbing chain is **not** regular (the absorbing state creates
a permanent zero row/column pattern). It still converges, but to a different
structure than a regular chain.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Key Takeaway |
|---|---|
| **Stochastic matrix** | Square, non-negative entries, columns sum to 1 |
| **State vector** | Column vector with non-negative entries summing to 1 |
| **Markov update** | $\mathbf{x}_{k+1} = P \mathbf{x}_k$ --- one matrix multiply per time step |
| **Matrix power** | $\mathbf{x}_k = P^k \mathbf{x}_0$ --- the $k$-step distribution |
| **Steady state** | $P\mathbf{q} = \mathbf{q}$ --- the eigenvector of $P$ for eigenvalue 1 |
| **Algebraic method** | Solve $(P - I)\mathbf{x} = \mathbf{0}$ with $\sum x_i = 1$ |
| **Regular chain** | Some $P^k$ all positive $\Rightarrow$ unique $\mathbf{q}$, convergence from any $\mathbf{x}_0$ |